# Phase VI — Head-to-head: Tarozo ordinal patterns vs multiscale level-set geometry

This phase compares the closest interpretable prior representation—Tarozo et al.'s 75 tie-aware two-by-two ordinal patterns—with our multiscale level-set curvature on the **same ArtBench pilot** and under the **same artist-disjoint nested-CV protocol**.

Primary question:

\[
\boxed{\text{Does multiscale level-set curvature add information beyond tie-aware ordinal patterns?}}
\]

The ordinary published accuracy from Tarozo et al. is **not** compared directly with ours because the datasets and validation protocols differ. Here the probe classifier and folds are held fixed so that the changing factor is the representation.


In [ ]:
import os, sys, subprocess, shutil, zipfile, urllib.request
from pathlib import Path

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# Robust repository bootstrap: shallow git clone first, then GitHub branch archive fallback.
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
    text=True, capture_output=True
)
if clone.returncode != 0:
    print("git clone failed (using archive fallback):")
    print(clone.stderr[-1500:])
    archive = Path("/content/painting-geometry-branch.zip")
    url = f"https://github.com/ardominguezm/painting-geometry/archive/refs/heads/{BRANCH}.zip"
    urllib.request.urlretrieve(url, archive)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall("/content")
    extracted = Path(f"/content/painting-geometry-{BRANCH}")
    if not extracted.exists():
        candidates = [p for p in Path('/content').glob('painting-geometry-*') if p.is_dir()]
        if len(candidates) != 1:
            raise RuntimeError(f"Could not identify extracted repository directory: {candidates}")
        extracted = candidates[0]
    extracted.rename(REPO_DIR)
    print("Repository loaded from GitHub branch archive.")
else:
    print("Repository cloned successfully.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ordpy>=1.2.0"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import pandas as pd, numpy as np, ordpy
print("Branch:", BRANCH)
if (REPO_DIR / ".git").exists():
    print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
else:
    print("Commit: branch archive fallback (no local .git metadata)")
print("ordpy:", getattr(ordpy, "__version__", "unknown"))
assert hasattr(ordpy, "two_by_two_patterns"), "ordpy>=1.2.0 is required"
print("Setup OK")


## 1. Upload previous outputs
Upload both files when prompted:

- `painting_geometry_phase4_artbench_pilot.zip`
- `painting_geometry_phase4b_scale_hierarchy.zip`

The first provides the exact 4,000-image pilot manifest/features. The second provides already-computed OOF predictions for curvature and the strong baseline, so they are not retrained.


In [ ]:
from google.colab import files
uploaded = files.upload()

needed = ["painting_geometry_phase4_artbench_pilot.zip", "painting_geometry_phase4b_scale_hierarchy.zip"]
missing = [x for x in needed if x not in uploaded]
if missing:
    raise FileNotFoundError(f"Missing required uploads: {missing}")

WORK = Path("/content/phase6")
if WORK.exists():
    shutil.rmtree(WORK)
P4 = WORK / "phase4"
P4B = WORK / "phase4b"
P4.mkdir(parents=True)
P4B.mkdir(parents=True)

for name, dest in [(needed[0], P4), (needed[1], P4B)]:
    with zipfile.ZipFile(Path('/content') / name) as zf:
        zf.extractall(dest)

def one_file(root, name):
    hits = list(root.rglob(name))
    if len(hits) != 1:
        raise RuntimeError(f"Expected exactly one {name} below {root}; found {hits}")
    return hits[0]

FEATURES = one_file(P4, "artbench_pilot_features.csv")
OOF_ALL = one_file(P4B, "artbench10_all_phase4b_oof_predictions.csv")
OOF_W8 = one_file(P4B, "artbench10_wikiart8_phase4b_oof_predictions.csv")
print("Features:", FEATURES)
print("OOF all10:", OOF_ALL)
print("OOF WikiArt8:", OOF_W8)
feat = pd.read_csv(FEATURES)
print("Pilot matrix:", feat.shape)
print("Styles:", sorted(feat['style'].unique()))


## 2. Obtain the ArtBench-10 256×256 ImageFolder
Ordinal patterns must be recomputed from pixels, so the image archive is required. The cell first attempts the Kaggle single-file download and otherwise uses the official ArtBench archive.


In [ ]:
import tarfile, kagglehub
DATA_DIR = Path("/content/artbench_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_HANDLE = "alexanderliao/artbench10"
TAR_NAME = "artbench-10-imagefolder-split.tar"
tar_path = None

for candidate in [TAR_NAME, f"256X256/{TAR_NAME}", f"data/256X256/{TAR_NAME}", f"ArtBench-10/data/256X256/{TAR_NAME}"]:
    try:
        print("Trying Kaggle file:", candidate)
        p = Path(kagglehub.dataset_download(KAGGLE_HANDLE, path=candidate))
        if p.exists():
            tar_path = p
            print("Downloaded:", p)
            break
    except Exception as exc:
        print("  not found:", type(exc).__name__)

if tar_path is None:
    tar_path = DATA_DIR / TAR_NAME
    official = "https://artbench.eecs.berkeley.edu/files/artbench-10-imagefolder-split.tar"
    print("Falling back to official ArtBench archive (~1.85 GB)...")
    subprocess.run(["wget", "-c", official, "-O", str(tar_path)], check=True)

EXTRACT_DIR = DATA_DIR / "imagefolder"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
train_hits = list(EXTRACT_DIR.rglob("train"))
if not train_hits:
    print("Extracting ArtBench...")
    with tarfile.open(tar_path) as tf:
        try:
            tf.extractall(EXTRACT_DIR, filter="data")
        except TypeError:
            tf.extractall(EXTRACT_DIR)
print("Image archive ready:", EXTRACT_DIR)


## 3. Extract Tarozo-style ordinal representations
For each of the same 4,000 pilot paintings we compute:

- `OP75`: 75 tie-aware 2×2 pattern probabilities;
- `OP11`: their 11 grouped types;
- `OP24`: conventional no-tie ordinal patterns;
- `HC`: permutation entropy and statistical complexity from the standard ordinal distribution.

We use `ordpy.two_by_two_patterns` from the same software lineage as the 2025 study.


In [ ]:
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
ENRICHED = RESULTS / "artbench_pilot_features_with_ordinal.csv"

subprocess.run([
    sys.executable, "-u", "scripts/extract_tarozo_ordinal_features.py",
    "--features", str(FEATURES),
    "--dataset-root", str(EXTRACT_DIR),
    "--output", str(ENRICHED),
    "--checkpoint-every", "250",
], check=True)

df = pd.read_csv(ENRICHED)
print("Enriched matrix:", df.shape)
print("OP75:", sum(c.startswith('ord75__') for c in df.columns))
print("OP11:", sum(c.startswith('ord11__') for c in df.columns))
print("OP24:", sum(c.startswith('ord24__') for c in df.columns))
print("HC:", sum(c.startswith('ordhc__') for c in df.columns))
for c in ['ordmeta__sum75','ordmeta__sum11','ordmeta__sum24','ordmeta__tie_pattern_mass','ordmeta__type_A_0000']:
    print(c, 'mean=', float(df[c].mean()), 'range=', (float(df[c].min()), float(df[c].max())))
assert np.allclose(df['ordmeta__sum75'], 1.0, atol=1e-8)
assert np.allclose(df['ordmeta__sum11'], 1.0, atol=1e-8)
assert np.allclose(df['ordmeta__sum24'], 1.0, atol=1e-8)
print("Ordinal sanity checks OK")


## 4. Artist-disjoint head-to-head
The central contrasts are:

\[
\Delta_1=F_1(OP75+K40)-F_1(OP75),
\]

and the exact dimension-matched version

\[
\Delta_2=F_1[(OP75+K40)_{k=40}]-F_1[(OP75)_{k=40}].
\]

For WikiArt-8 we additionally test whether curvature still contributes after the strong conventional baseline **and** OP75 are already present.


In [ ]:
EXP = RESULTS / "head_to_head"
EXP.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, "-u", "scripts/run_ordinal_geometry_head_to_head.py",
    "--features", str(ENRICHED),
    "--output-dir", str(EXP),
    "--phase4b-all-oof", str(OOF_ALL),
    "--phase4b-wiki8-oof", str(OOF_W8),
    "--outer-folds", "5",
    "--inner-folds", "3",
    "--n-jobs", "-1",
    "--metric-boot", "2000",
    "--delta-boot", "5000",
], check=True)

res = pd.read_csv(EXP / "phase6_head_to_head_results.csv")
delr = pd.read_csv(EXP / "phase6_head_to_head_deltas.csv")
print("\nMODEL RESULTS")
display(res.sort_values(['dataset','macro_f1_oof'], ascending=[True,False]))
print("\nPAIRED DELTAS")
display(delr)
print("\nPRIMARY HEAD-TO-HEAD")
display(delr[delr['primary_head_to_head'].astype(bool)])


## 5. Package outputs
Download the ZIP and upload it back to the chat for interpretation. The conclusions will be based primarily on artist-group bootstrap CIs and the prespecified geometry-increment contrasts, not on raw accuracy alone.


In [ ]:
OUT_ZIP = Path("/content/painting_geometry_phase6_ordinal_head_to_head.zip")
if OUT_ZIP.exists(): OUT_ZIP.unlink()
with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(RESULTS))
print("Output ZIP:", OUT_ZIP, "size MB:", OUT_ZIP.stat().st_size/1e6)
files.download(str(OUT_ZIP))
